# 03. Disclosure Text — 공시 텍스트 분석

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/dartlab/blob/master/notebooks/tutorials/03_disclosure.ipynb)

숫자 뒤에 숨겨진 텍스트를 읽는다.

**학습 내용:**
- 사업의 내용 (섹션별 분류, 변경 탐지)
- 경영진단 및 분석의견 (MD&A)
- 회사의 개요 (설립일, 신용등급)
- 원재료·설비투자 현황

In [ ]:
!pip install -q dartlab

In [ ]:
from dartlab import Company

samsung = Company("005930")

## 1. 사업의 내용

사업보고서의 '사업의 내용' 섹션을 구조적으로 분류한다.

In [ ]:
biz = samsung.business()

print(f"기업: {biz.corpName}, 연도: {biz.year}")
print(f"섹션 수: {len(biz.sections)}")
print()
for section in biz.sections:
    print(f"[{section.key}] {section.title} ({section.chars:,}자)")

In [ ]:
overview_section = next((s for s in biz.sections if s.key == "overview"), None)
if overview_section:
    print(overview_section.text[:500])

### 연도별 변경 탐지

텍스트 변경량을 연도별로 추적한다.

In [ ]:
for change in biz.changes:
    print(f"{change.year}: 변경 {change.changedPct:.1%} | +{change.added:,}자 -{change.removed:,}자 | 총 {change.totalChars:,}자")

## 2. 경영진단 및 분석의견 (MD&A)

이사의 경영진단 및 분석의견을 섹션별로 분류한다.

In [ ]:
mdna = samsung.mdna()

print(f"섹션 수: {len(mdna.sections)}")
print()
for section in mdna.sections:
    print(f"[{section.category}] {section.title}")
    print(f"  텍스트 {section.textLines}줄, 테이블 {section.tableLines}줄")

In [ ]:
if mdna.overview:
    print("=== 사업 개요 ===")
    print(mdna.overview[:500])

## 3. 회사의 개요

설립일, 주소, 신용등급 등 정량 데이터를 추출한다.

In [ ]:
ov = samsung.overview()

if ov:
    print(f"설립: {ov.founded}")
    print(f"소재지: {ov.address}")
    print(f"홈페이지: {ov.homepage}")
    print(f"종속회사: {ov.subsidiaryCount}개")
    print(f"중소기업: {ov.isSME}")
    print(f"상장일: {ov.listedDate}")
    print()
    print("--- 신용등급 ---")
    for cr in ov.creditRatings:
        print(f"  {cr.agency}: {cr.grade}")

## 4. 원재료·설비투자

In [ ]:
rm = samsung.rawMaterial()

if rm:
    print("--- 원재료 매입 ---")
    for m in rm.materials[:5]:
        print(f"  {m.item}: {m.amount}백만원 ({m.ratio}%)")

    if rm.equipment:
        eq = rm.equipment
        print(f"\n--- 유형자산 ---")
        print(f"  합계: {eq.total}, CAPEX: {eq.capex}")

    print(f"\n--- 설비투자 ---")
    for item in rm.capexItems:
        print(f"  {item.segment}: {item.amount}백만원")